In [ ]:
# pip install -U langchain langgraph langchain-openai langchain-chroma

import os

from typing import TypedDict

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

from langgraph.graph import StateGraph, START, END


# ---------------------------------------------------------
# 1. LLM
# ---------------------------------------------------------

llm = ChatOllama(model="llama3.2", temperature=0)

embeddings = OllamaEmbeddings(model="nomic-embed-text")




In [ ]:
# ---------------------------------------------------------
# 2. Sample knowledge base
# ---------------------------------------------------------

documents = [
    Document(
        page_content="""
        LangGraph is a framework for building stateful,
        multi-step applications with LLMs. It represents
        workflows as graphs containing nodes and edges.
        """
    ),

    Document(
        page_content="""
        LangGraph nodes contain the logic of individual
        steps in a workflow. A node can call an LLM,
        execute a tool, retrieve documents, or perform
        custom processing.
        """
    ),

    Document(
        page_content="""
        LangGraph supports persistence and memory.
        Checkpoints can be used to save graph state so
        that an application can resume execution.
        """
    ),

    Document(
        page_content="""
        LangChain provides components for building
        LLM applications, including document loaders,
        text splitters, embeddings, vector stores,
        retrievers, prompts and chains.
        """
    ),

    Document(
        page_content="""
        RAG, or Retrieval Augmented Generation, retrieves
        relevant information from an external knowledge
        base and provides that information to an LLM
        before generating an answer.
        """
    )
]


In [ ]:
# ---------------------------------------------------------
# 3. Create vector store
# ---------------------------------------------------------

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:

# ---------------------------------------------------------
# 4. Define Graph State
# ---------------------------------------------------------

class RAGState(TypedDict):

    original_query: str
    rewritten_query: str
    documents: list
    answer: str


In [ ]:
# ---------------------------------------------------------
# 5. Query Rewriting Node
# ---------------------------------------------------------

def rewrite_query(state: RAGState):

    original_query = state["original_query"]

    prompt = f"""
You are a search query optimizer.

Rewrite the user's question into a concise query
that will work well for semantic search over a
technical knowledge base.

Remove unnecessary conversational words.

Preserve the important technical concepts.

User question:
{original_query}

Return ONLY the rewritten search query.
"""

    response = llm.invoke(prompt)

    rewritten_query = response.content.strip()

    print("\nOriginal query:")
    print(original_query)

    print("\nRewritten query:")
    print(rewritten_query)

    return {
        "rewritten_query": rewritten_query
    }


In [ ]:
# ---------------------------------------------------------
# 6. Retrieval Node
# ---------------------------------------------------------

def retrieve_documents(state: RAGState):

    query = state["rewritten_query"]

    docs = retriever.invoke(query)

    print("\nRetrieved Documents:")

    for i, doc in enumerate(docs):
        print(f"\nDocument {i + 1}:")
        print(doc.page_content)

    return {
        "documents": docs
    }


In [ ]:
# ---------------------------------------------------------
# 7. Answer Generation Node
# ---------------------------------------------------------

def generate_answer(state: RAGState):

    query = state["original_query"]
    docs = state["documents"]

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    prompt = f"""
You are a helpful technical assistant.

Answer the user's question using ONLY the
provided context.

If the answer is not present in the context,
say that you don't have enough information.

Context:
{context}

User Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }


In [ ]:
# ---------------------------------------------------------
# 8. Build LangGraph
# ---------------------------------------------------------

graph = StateGraph(RAGState)

graph.add_node(
    "rewrite_query",
    rewrite_query
)

graph.add_node(
    "retrieve",
    retrieve_documents
)

graph.add_node(
    "generate_answer",
    generate_answer
)


# ---------------------------------------------------------
# 9. Define edges
# ---------------------------------------------------------

graph.add_edge(
    START,
    "rewrite_query"
)

graph.add_edge(
    "rewrite_query",
    "retrieve"
)

graph.add_edge(
    "retrieve",
    "generate_answer"
)

graph.add_edge(
    "generate_answer",
    END
)


# ---------------------------------------------------------
# 10. Compile
# ---------------------------------------------------------

app = graph.compile()


In [ ]:
# ---------------------------------------------------------
# 11. Execute
# ---------------------------------------------------------

result = app.invoke({
    "original_query":
        "What does LangGraph use to save what happened earlier?"
})


print("\n\nFINAL ANSWER:")
print(result["answer"])